# 21. Transformer Encoder 직접 구현

> **제21장** · **이론편 대응: 18장 (Transformer 아키텍처)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **다운로드**: 없음

---

## 이 장에서 하는 일

20장에서 Attention의 네 단계를 확인했다. 그런데 Attention만으로는 Transformer가 되지 않는다.
**부품을 하나씩 얹어 인코더 블록을 완성한다.**

| 절 | 부품 | 왜 필요한가 | 이론편 |
|---|---|---|---|
| 1 | **Positional Encoding** | Attention은 순서를 모른다 | 18.2절 |
| 2 | **Multi-Head Attention** | 여러 관계를 동시에 | 18.4절 |
| 3 | **Feed-Forward** | 토큰별 비선형 변환 | 18.5절 |
| 4 | **Residual 연결** | 깊이 쌓아도 학습되게 | 18.5절 |
| 5 | **Layer Normalization** | 학습 안정화 | 18.5절 |
| 6 | 블록 조립 | | 18.5절 |
| 7 | 실제로 학습시켜 보기 | | |

**1절부터 순서대로 "왜 이게 필요한가"를 먼저 확인**하고 부품을 붙인다.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

---

## 1. Positional Encoding — 이론편 18.2절

### 문제 확인부터

20장에서 만든 Attention에 **순서를 바꾼 입력**을 넣어 보자. 결과가 어떻게 될까.

In [ ]:
import numpy as np


def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V):
    """20장에서 만든 것과 같다"""
    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = softmax(scores)
    return weights @ V, weights


# 세 토큰
X = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])

out_original, _ = attention(X, X, X)

# 순서를 뒤집은 입력
X_shuffled = X[[2, 1, 0]]
out_shuffled, _ = attention(X_shuffled, X_shuffled, X_shuffled)

print("=" * 60)
print("Attention은 순서를 아는가")
print("=" * 60)
print("원래 순서 (토큰 0, 1, 2)")
print(out_original)
print()
print("뒤집은 순서 (토큰 2, 1, 0)")
print(out_shuffled)
print()
print("뒤집은 결과를 다시 원래대로 돌려놓으면")
print(out_shuffled[[2, 1, 0]])
print("-" * 60)

same = np.allclose(out_original, out_shuffled[[2, 1, 0]])
print(f"원래 결과와 같은가: {same}")
assert same
print()
print("[문제] Attention은 순서를 전혀 모른다.")
print("  각 토큰 쌍의 관계만 볼 뿐, 누가 먼저인지 신경 쓰지 않는다.")
print()
print("  '나는 너를 좋아해' 와 '너를 나는 좋아해' 가 같아진다.")
print("  RNN(16장)은 순차 처리라 순서를 자연히 알았지만, Attention은 그렇지 않다.")

### 해법 — 위치 정보를 더한다

이론편 18.2절의 해법은 **각 위치마다 고유한 벡터를 만들어 입력에 더하는 것**이다.

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad
PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

식이 복잡해 보이지만 하는 일은 단순하다. **차원마다 다른 주기의 파동**을 만들어,
위치마다 고유한 무늬가 나오게 하는 것이다.

In [ ]:
import numpy as np


def positional_encoding(max_len, d_model):
    """사인·코사인 위치 인코딩 (이론편 18.2절)

    짝수 차원에는 sin, 홀수 차원에는 cos를 넣는다.
    차원이 커질수록 주기가 길어진다.
    """
    pos = np.arange(max_len)[:, np.newaxis]        # (max_len, 1)
    i = np.arange(d_model)[np.newaxis, :]          # (1, d_model)

    # 차원에 따라 주기가 달라진다
    angle_rates = 1.0 / np.power(10000, (2 * (i // 2)) / d_model)
    angles = pos * angle_rates

    PE = np.zeros((max_len, d_model))
    PE[:, 0::2] = np.sin(angles[:, 0::2])          # 짝수 차원
    PE[:, 1::2] = np.cos(angles[:, 1::2])          # 홀수 차원
    return PE


PE = positional_encoding(max_len=10, d_model=8)

print("=" * 60)
print("Positional Encoding")
print("=" * 60)
print(f"모양: {PE.shape}   ← (위치, 차원)")
print()
print("위치 0의 벡터")
print(f"  {PE[0]}")
print("  sin(0)=0, cos(0)=1 이 번갈아 나온다")
assert np.allclose(PE[0], [0, 1, 0, 1, 0, 1, 0, 1])
print()
print("위치 1의 벡터")
print(f"  {PE[1]}")
print()
print("앞 4개 위치")
for p in range(4):
    print(f"  pos={p}: {PE[p]}")
print()
print("위치마다 다른 무늬가 나온다 — 이것이 위치를 구별하는 신호가 된다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PE_big = positional_encoding(max_len=50, d_model=64)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 전체 무늬 ---
ax = axes[0]
im = ax.imshow(PE_big, cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xlabel("차원")
ax.set_ylabel("위치")
ax.set_title("Positional Encoding 전체 무늬")

# --- 오른쪽: 몇 개 차원의 파동 ---
ax = axes[1]
for d in [0, 2, 8, 20]:
    ax.plot(PE_big[:, d], label=f"차원 {d}", linewidth=2)
ax.set_xlabel("위치")
ax.set_ylabel("값")
ax.set_title("차원마다 다른 주기")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("왼쪽: 위치가 달라지면 무늬가 달라진다")
print("오른쪽: 낮은 차원은 빠르게, 높은 차원은 천천히 진동한다")
print()
print("빠른 파동은 가까운 위치를 구별하고,")
print("느린 파동은 멀리 떨어진 위치를 구별한다.")
print("→ 시계의 초침·분침·시침과 비슷한 원리 (이론편 18.2절)")

In [ ]:
import numpy as np

print("=" * 60)
print("위치를 더하면 순서를 알게 되는가")
print("=" * 60)

d_model = 2
X_pos = X + positional_encoding(3, d_model)      # 위치 정보를 더한다

out_with_pos, _ = attention(X_pos, X_pos, X_pos)

# 순서를 바꾼 경우
X_shuf_pos = X[[2, 1, 0]] + positional_encoding(3, d_model)
out_shuf_pos, _ = attention(X_shuf_pos, X_shuf_pos, X_shuf_pos)

print("위치 정보를 더한 뒤")
print("  원래 순서:")
print(out_with_pos)
print("  뒤집은 순서를 되돌린 것:")
print(out_shuf_pos[[2, 1, 0]])
print("-" * 60)

same_now = np.allclose(out_with_pos, out_shuf_pos[[2, 1, 0]])
print(f"같은가: {same_now}")
assert not same_now
print()
print("[해결] 이제 순서가 바뀌면 결과도 달라진다.")
print("  Attention 자체는 그대로인데, 입력에 위치 신호가 섞였기 때문이다.")

### 왜 학습하지 않고 고정된 식을 쓰는가

위치 벡터를 학습시킬 수도 있다(실제로 그런 모델도 있다). 사인·코사인을 쓰는 이유는 이론편 18.2절에서 다뤘다.

1. **학습 때 본 적 없는 긴 문장에도 적용된다** — 식으로 계산하면 되므로
2. **상대적 위치 관계가 자연스럽게 표현된다** — 삼각함수의 덧셈정리 덕분에

두 번째를 확인해 보자.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PE_test = positional_encoding(max_len=50, d_model=64)

# 위치 10을 기준으로 다른 위치와의 유사도
ref = 10
sims = [PE_test[ref] @ PE_test[p] for p in range(50)]

print("=" * 55)
print(f"위치 {ref}과 다른 위치의 내적")
print("=" * 55)
for p in [8, 9, 10, 11, 12, 20, 40]:
    mark = "  ← 자기 자신" if p == ref else ""
    print(f"  위치 {p:2}: {sims[p]:8.3f}{mark}")
print()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sims, linewidth=2, color="#1E40AF")
ax.axvline(ref, color="#EA580C", linestyle="--", linewidth=1.5)
ax.text(ref + 1, max(sims) * 0.9, f"기준 위치 {ref}", color="#EA580C", fontsize=9)
ax.set_xlabel("위치")
ax.set_ylabel("내적 (유사도)")
ax.set_title(f"위치 {ref}과 각 위치의 유사도")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("가까운 위치일수록 유사도가 높다.")
print("모델이 '이 둘은 가깝다/멀다'를 판단할 근거가 생기는 것이다.")

---

## 2. Multi-Head Attention — 이론편 18.4절

20장 9절에서 개념만 봤다. 이번에는 제대로 구현한다.

**20장과 달라지는 점**: 각 헤드마다 **별도의 가중치 행렬** $W_Q, W_K, W_V$를 둔다.
입력을 그냥 잘라 쓰는 것이 아니라, 헤드마다 다른 방식으로 변환한 뒤 Attention을 계산한다.

$$\text{head}_i = \text{Attention}(XW_Q^i,\ XW_K^i,\ XW_V^i)$$

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W_O$$

마지막에 $W_O$를 곱해 여러 헤드의 결과를 다시 섞는다.

In [ ]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    # Multi-Head Attention (이론편 18.4절)

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model이 헤드 수로 나누어떨어져야 합니다"

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        # 네 개의 선형 변환 (헤드별로 나누지 않고 한 번에 처리)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        # (배치, 토큰, d_model) → (배치, 헤드, 토큰, d_head)
        b, t, _ = x.shape
        x = x.view(b, t, self.n_heads, self.d_head)
        return x.transpose(1, 2)

    def forward(self, x, mask=None, return_weights=False):
        b, t, _ = x.shape

        # 1) 선형 변환 후 헤드로 나눈다
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # 2) Attention (20장에서 만든 네 단계)
        scores = (Q @ K.transpose(-2, -1)) / (self.d_head ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        out = weights @ V

        # 3) 헤드를 다시 이어 붙인다
        out = out.transpose(1, 2).contiguous().view(b, t, self.d_model)

        # 4) 마지막 선형 변환
        out = self.W_o(out)

        if return_weights:
            return out, weights
        return out


print("=" * 60)
print("Multi-Head Attention")
print("=" * 60)

d_model, n_heads = 64, 4
mha = MultiHeadAttention(d_model, n_heads)

x = torch.randn(2, 5, d_model)      # 배치2, 토큰5
out, w = mha(x, return_weights=True)

print(f"d_model    : {d_model}")
print(f"헤드 개수  : {n_heads}")
print(f"헤드당 차원: {d_model // n_heads}")
print()
print(f"입력   : {tuple(x.shape)}     ← (배치, 토큰, d_model)")
print(f"가중치 : {tuple(w.shape)}  ← (배치, 헤드, 토큰, 토큰)")
print(f"출력   : {tuple(out.shape)}     ← 입력과 같은 모양")
print()
print(f"파라미터: {sum(p.numel() for p in mha.parameters()):,}개")
print(f"  계산: 4 x (64x64 + 64) = {4*(64*64+64):,}")
assert sum(p.numel() for p in mha.parameters()) == 4*(64*64+64)
print("[OK] W_q, W_k, W_v, W_o 네 개")

### `contiguous()`가 필요한 이유

`transpose(1, 2)` 뒤에 `.contiguous()`를 호출했다. 이유가 있다.

`transpose`는 실제로 데이터를 옮기지 않고 **읽는 순서만 바꾼다.** 메모리에는 원래 순서로 남아 있다.
그런데 `view`는 메모리가 연속으로 배치되어 있다고 가정하므로, 그대로 부르면 오류가 난다.

`contiguous()`가 메모리를 실제로 재배치해 준다. 이 순서를 빠뜨리면 다음 오류를 만난다.

```
RuntimeError: view size is not compatible with input tensor's size and stride
```

`reshape()`를 쓰면 필요할 때 알아서 복사하므로 이 문제가 없다. 다만 언제 복사가 일어나는지
명확하지 않으므로, 학습용으로는 `contiguous().view()`가 의도를 드러내기에 낫다.

---

## 3. Feed-Forward Network — 이론편 18.5절

Attention은 **토큰들 사이의 정보를 섞는** 일을 한다. 그런데 각 토큰 자체를 변환하는 부분이 없다.

FFN이 그 역할이다. **모든 토큰에 같은 신경망을 따로 적용**한다.

$$\text{FFN}(x) = W_2\,\text{ReLU}(W_1 x + b_1) + b_2$$

특징은 **가운데를 넓게** 만든다는 것이다. 보통 $d_{model}$의 4배로 늘렸다가 다시 줄인다.

In [ ]:
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    # 위치별 Feed-Forward (이론편 18.5절)

    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or d_model * 4        # 관행적으로 4배
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


print("=" * 60)
print("Feed-Forward Network")
print("=" * 60)

d_model = 64
ffn = FeedForward(d_model)

x = torch.randn(2, 5, d_model)
out = ffn(x)

print(f"입력 : {tuple(x.shape)}")
print(f"중간 : (2, 5, {d_model*4})   ← 4배로 확장")
print(f"출력 : {tuple(out.shape)}   ← 다시 원래 크기")
print()
print(f"파라미터: {sum(p.numel() for p in ffn.parameters()):,}개")
print()

# 토큰별로 독립적인지 확인
print("각 토큰에 독립적으로 적용되는가")
x_single = x[0:1, 0:1, :]                 # 토큰 하나만
out_single = ffn(x_single)
print(f"  전체 처리 시 첫 토큰 출력: {out[0,0,:4].detach().numpy().round(4)}")
print(f"  토큰 하나만 넣었을 때    : {out_single[0,0,:4].detach().numpy().round(4)}")
same = torch.allclose(out[0,0], out_single[0,0], atol=1e-5)
print(f"  같은가: {same}")
print()
print("→ FFN은 토큰끼리 섞지 않는다. 정보를 섞는 것은 Attention의 역할이다.")
print()
print("역할 분담")
print("  Attention : 토큰들 사이의 정보 교환")
print("  FFN       : 각 토큰의 표현을 변환")

---

## 4. Residual 연결 — 이론편 18.5절

이제 부품이 두 개(Attention, FFN) 생겼다. 그냥 쌓으면 될까?

**깊이 쌓으면 학습이 안 된다.** 이론편 11.2절에서 다룬 그래디언트 소실 때문이다.
11장에서 층을 거칠 때마다 그래디언트가 줄어드는 것을 직접 측정했었다.

**해법은 단순하다.** 입력을 출력에 그대로 더한다.

$$y = x + F(x)$$

이러면 미분할 때 $\frac{\partial y}{\partial x} = 1 + \frac{\partial F}{\partial x}$가 되어,
$F$의 미분이 아무리 작아도 **1이 남는다.** 그래디언트가 통과할 길이 생기는 것이다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 60)
print("Residual 연결의 효과 — 그래디언트 측정")
print("=" * 60)


class PlainBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return self.net(x)


class ResidualBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return x + self.net(x)          # 입력을 더한다


def measure_gradient(block_cls, n_layers, d=32, seed=0):
    torch.manual_seed(seed)
    layers = nn.Sequential(*[block_cls(d) for _ in range(n_layers)])
    x = torch.randn(1, d, requires_grad=True)
    out = layers(x)
    out.sum().backward()
    return x.grad.abs().mean().item()


print(f"{'층 수':<10}{'Residual 없음':<22}{'Residual 있음':<22}{'비율'}")
print("-" * 60)
for n in [2, 5, 10, 20]:
    g_plain = measure_gradient(PlainBlock, n)
    g_res = measure_gradient(ResidualBlock, n)
    ratio = g_res / g_plain if g_plain > 0 else float("inf")
    print(f"{n:<10}{g_plain:<22.3e}{g_res:<22.3e}{ratio:>10.1f}배")

print("-" * 60)
print()
print("층이 깊어질수록 차이가 벌어진다.")
print("Residual 없이는 20층에서 그래디언트가 거의 사라진다.")
print()
print("이것이 이론편 18.5절에서 '깊이 쌓을 수 있게 된 이유'로 든 것이다.")
print("오늘날 LLM이 수십~수백 층을 쌓을 수 있는 것도 이 덕분이다.")

---

## 5. Layer Normalization — 이론편 18.5절

마지막 부품이다. 이론편 11.3절에서 Batch Normalization을 다뤘는데,
Transformer는 **Layer Normalization**을 쓴다.

**차이는 "무엇을 기준으로 정규화하는가"**다.

| | 정규화 기준 | 배치 크기 의존 |
|---|---|---|
| BatchNorm | 배치 안의 같은 특성끼리 (열 방향) | 있음 |
| **LayerNorm** | 한 샘플의 모든 특성끼리 (행 방향) | **없음** |

직접 비교해 보자.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 60)
print("LayerNorm vs BatchNorm")
print("=" * 60)

X = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                  [10.0, 20.0, 30.0, 40.0]])
print("입력 (샘플 2개, 특성 4개)")
print(X.numpy())
print()

ln = nn.LayerNorm(4, elementwise_affine=False)
bn = nn.BatchNorm1d(4, affine=False)
bn.train()

print("LayerNorm — 각 행을 따로 정규화")
print(ln(X).numpy().round(3))
print("  → 두 행이 같은 값이 되었다 (비율이 같으므로)")
print()
print("BatchNorm — 각 열을 따로 정규화")
print(bn(X).detach().numpy().round(3))
print("  → 각 열에서 두 값이 -1, 1이 되었다")
print()
print("-" * 60)
print("Transformer가 LayerNorm을 쓰는 이유")
print("  1) 문장 길이가 제각각이라 배치 통계가 불안정하다")
print("  2) 생성할 때는 토큰을 하나씩 만드는데, 배치가 1이면 BatchNorm이 무의미하다")
print("  3) 각 샘플 안에서만 계산하므로 배치 크기와 무관하다")

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 55)
print("LayerNorm을 손으로 계산")
print("=" * 55)

x = np.array([1.0, 2.0, 3.0, 4.0])
mean = x.mean()
std = x.std()

print(f"입력   : {x}")
print(f"평균   : {mean}")
print(f"표준편차: {std:.6f}")
print()
print("정규화: (x - 평균) / 표준편차")
normalized = (x - mean) / std
for xi, ni in zip(x, normalized):
    print(f"  ({xi} - {mean}) / {std:.4f} = {ni:.6f}")
print()
print(f"결과: {normalized.round(6)}")
print(f"평균: {normalized.mean():.6f}   표준편차: {normalized.std():.6f}")
print()

# PyTorch와 대조
ln = nn.LayerNorm(4, elementwise_affine=False)
torch_result = ln(torch.tensor(x, dtype=torch.float32)).numpy()
print(f"PyTorch: {torch_result.round(6)}")
print("-" * 55)
assert np.allclose(normalized, torch_result, atol=1e-4)
print("[OK] 손계산과 일치")
print()
print("실제 LayerNorm에는 학습 가능한 scale과 shift가 붙는다.")
print("  y = gamma * normalized + beta")
print("  정규화가 오히려 해로울 때 모델이 되돌릴 수 있게 하는 장치다 (이론편 11.3절).")

---

## 6. 블록 조립 — 이론편 18.5절

네 부품이 모두 준비됐다. 이제 순서대로 연결한다.

```
입력
 ├─────────────┐
 │             │
 ↓             │
Multi-Head     │  (Residual)
Attention      │
 ↓             │
 + ←───────────┘
 ↓
LayerNorm
 ├─────────────┐
 │             │
 ↓             │
Feed-Forward   │  (Residual)
 ↓             │
 + ←───────────┘
 ↓
LayerNorm
 ↓
출력
```

**Residual → LayerNorm 순서**가 원 논문 방식(Post-LN)이다.
최근에는 LayerNorm을 먼저 하는 Pre-LN도 많이 쓰는데, 7절에서 비교한다.

In [ ]:
import torch
import torch.nn as nn


class EncoderBlock(nn.Module):
    # Transformer 인코더 블록 (이론편 18.5절)

    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1) Attention + Residual + Norm
        attn_out = self.attn(x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # 2) FFN + Residual + Norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x


print("=" * 60)
print("인코더 블록")
print("=" * 60)

d_model, n_heads = 64, 4
block = EncoderBlock(d_model, n_heads)

x = torch.randn(2, 5, d_model)
out = block(x)

print(f"입력: {tuple(x.shape)}")
print(f"출력: {tuple(out.shape)}   ← 같은 모양이라 계속 쌓을 수 있다")
print()

total = sum(p.numel() for p in block.parameters())
print("파라미터 구성")
print(f"  Multi-Head Attention : {sum(p.numel() for p in block.attn.parameters()):,}")
print(f"  Feed-Forward         : {sum(p.numel() for p in block.ffn.parameters()):,}")
print(f"  LayerNorm x 2        : {sum(p.numel() for p in block.norm1.parameters())*2:,}")
print(f"  {'합계':<21}: {total:,}")
print()

# PyTorch 내장 구현과 파라미터 수 비교
builtin = nn.TransformerEncoderLayer(
    d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, batch_first=True)
print(f"PyTorch 내장 구현: {sum(p.numel() for p in builtin.parameters()):,}개")
print(f"직접 구현        : {total:,}개")
print(f"일치: {total == sum(p.numel() for p in builtin.parameters())}")

In [ ]:
import torch
import torch.nn as nn


class TransformerEncoder(nn.Module):
    # 여러 블록을 쌓은 인코더 (이론편 18.5절)

    def __init__(self, vocab_size, d_model=64, n_heads=4,
                 n_layers=2, max_len=100, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # 토큰 → 벡터 (이론편 20.3절 Embedding)
        self.embedding = nn.Embedding(vocab_size, d_model)

        # 위치 정보 (1절에서 만든 것)
        pe = torch.tensor(positional_encoding(max_len, d_model), dtype=torch.float32)
        self.register_buffer("pe", pe)      # 학습 대상이 아니므로 buffer로

        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids, mask=None, return_attn=False):
        t = token_ids.shape[1]

        # 임베딩 + 위치 인코딩
        x = self.embedding(token_ids) * (self.d_model ** 0.5)
        x = x + self.pe[:t].unsqueeze(0)
        x = self.dropout(x)

        attns = []
        for block in self.blocks:
            if return_attn:
                a, w = block.attn(x, mask, return_weights=True)
                attns.append(w)
            x = block(x, mask)

        if return_attn:
            return x, attns
        return x


print("=" * 60)
print("전체 인코더")
print("=" * 60)

VOCAB = 100
encoder = TransformerEncoder(vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=2)

tokens = torch.randint(0, VOCAB, (2, 7))
out = encoder(tokens)

print(f"입력 토큰 : {tuple(tokens.shape)}   ← (배치, 토큰 수)")
print(f"출력      : {tuple(out.shape)}  ← (배치, 토큰 수, d_model)")
print()
print(f"전체 파라미터: {sum(p.numel() for p in encoder.parameters()):,}개")
print()
print("임베딩에 sqrt(d_model)을 곱하는 이유")
print("  위치 인코딩의 값이 -1~1인데 임베딩은 초기에 작으므로,")
print("  크기를 맞춰 주지 않으면 위치 정보가 지배해 버린다. (원 논문의 방식)")

---

## 7. 실제로 학습시켜 보기

부품이 제대로 동작하는지 확인하려면 **실제 과제를 풀려 봐야 한다.**

간단한 과제를 만든다 — **숫자 열에서 최댓값이 몇 번째에 있는지 맞히기.**
순서를 알아야 풀 수 있으므로 Positional Encoding이 작동하는지도 함께 확인된다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 과제: 길이 8인 수열에서 최댓값의 위치를 맞히기
SEQ_LEN = 8
VOCAB = 20

def make_data(n, seed=0):
    rng = np.random.RandomState(seed)
    X = rng.randint(1, VOCAB, size=(n, SEQ_LEN))
    y = X.argmax(axis=1)            # 최댓값의 위치
    return torch.tensor(X), torch.tensor(y)

X_train, y_train = make_data(3000, seed=0)
X_test, y_test = make_data(500, seed=1)

print("=" * 55)
print("과제: 수열에서 최댓값의 위치 맞히기")
print("=" * 55)
print("예시")
for i in range(3):
    print(f"  {X_train[i].numpy()} → 정답 {y_train[i].item()}번째 "
          f"(값 {X_train[i].max().item()})")
print()
print(f"학습 데이터: {len(X_train):,}개")
print(f"시험 데이터: {len(X_test):,}개")
print(f"무작위로 찍으면 정확도 {1/SEQ_LEN:.3f}")
print()
print("이 과제는 순서를 알아야 풀 수 있다.")
print("→ Positional Encoding이 제대로 작동하는지 확인할 수 있다.")

In [ ]:
import torch
import torch.nn as nn
import time


class MaxPositionModel(nn.Module):
    def __init__(self, use_pe=True, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.use_pe = use_pe
        self.d_model = d_model
        self.embedding = nn.Embedding(VOCAB, d_model)
        pe = torch.tensor(positional_encoding(SEQ_LEN, d_model), dtype=torch.float32)
        self.register_buffer("pe", pe)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, dropout=0.0) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, SEQ_LEN)

    def forward(self, tokens):
        x = self.embedding(tokens) * (self.d_model ** 0.5)
        if self.use_pe:
            x = x + self.pe.unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        return self.head(x.mean(dim=1))      # 토큰 평균 → 분류


def train_model(use_pe, epochs=30, seed=42):
    torch.manual_seed(seed)
    model = MaxPositionModel(use_pe=use_pe).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    Xtr, ytr = X_train.to(device), y_train.to(device)
    Xte, yte = X_test.to(device), y_test.to(device)

    accs = []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 128):
            idx = perm[i:i+128]
            opt.zero_grad()
            crit(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            acc = (model(Xte).argmax(1) == yte).float().mean().item()
        accs.append(acc)
    return model, accs


print("=" * 60)
print("Positional Encoding 유무 비교")
print("=" * 60)
t0 = time.time()

results = {}
for use_pe, name in [(True, "PE 있음"), (False, "PE 없음")]:
    model, accs = train_model(use_pe)
    results[name] = accs
    print(f"  {name:<12} 최종 정확도 {accs[-1]:.4f}   ({time.time()-t0:.0f}초)")

print("-" * 60)
print(f"무작위 수준: {1/SEQ_LEN:.4f}")
print()
print("PE가 없으면 순서를 모르므로 '몇 번째'를 맞힐 수 없다.")
print("1절에서 확인한 문제가 실제 학습에서 그대로 나타난 것이다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4.5))

for name, accs in results.items():
    ax.plot(accs, label=name, linewidth=2)

ax.axhline(1/SEQ_LEN, color="gray", linestyle="--", linewidth=1.5)
ax.text(1, 1/SEQ_LEN + 0.02, "무작위 수준", fontsize=9, color="gray")
ax.set_xlabel("에폭")
ax.set_ylabel("시험 정확도")
ax.set_title("Positional Encoding의 효과")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 학습된 Attention 들여다보기

모델이 무엇을 보고 판단했는지 확인한다. **최댓값이 있는 위치에 주의가 몰려 있을까?**

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

model_pe, _ = train_model(use_pe=True)
model_pe.eval()

# 시험 데이터 하나를 골라 Attention 확인
sample_idx = 0
sample = X_test[sample_idx:sample_idx+1].to(device)
answer = y_test[sample_idx].item()

with torch.no_grad():
    x = model_pe.embedding(sample) * (model_pe.d_model ** 0.5)
    x = x + model_pe.pe.unsqueeze(0)
    _, w = model_pe.blocks[0].attn(x, return_weights=True)

attn = w[0].cpu().numpy()      # (헤드, 토큰, 토큰)

print("=" * 55)
print("학습된 Attention 확인")
print("=" * 55)
print(f"입력: {sample[0].cpu().numpy()}")
print(f"정답: {answer}번째 (값 {sample[0].max().item()})")
print()

# 각 헤드가 최댓값 위치를 얼마나 보는지
print(f"{'헤드':<8}{'최댓값 위치에 준 평균 주의':<28}{'균등 대비'}")
print("-" * 55)
uniform = 1.0 / SEQ_LEN
for h in range(attn.shape[0]):
    to_max = attn[h][:, answer].mean()
    print(f"{h:<8}{to_max:<28.4f}{to_max/uniform:.2f}배")
print("-" * 55)
print(f"균등 배분 시: {uniform:.4f}")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for h, ax in enumerate(axes):
    im = ax.imshow(attn[h], cmap="Blues")
    ax.axvline(answer, color="#EA580C", linewidth=2, alpha=0.7)
    ax.set_title(f"헤드 {h}", fontsize=10)
    ax.set_xlabel("보는 대상")
    if h == 0:
        ax.set_ylabel("보는 주체")
    ax.tick_params(labelsize=7)

fig.suptitle(f"학습된 Attention (주황선 = 정답 위치 {answer})", fontsize=12)
plt.tight_layout()
plt.show()

print()
print("일부 헤드가 정답 위치 열에 주의를 몰아주고 있다면,")
print("모델이 '어디에 큰 값이 있는가'를 Attention으로 찾아냈다는 뜻이다.")

---

## 8. Pre-LN과 Post-LN

6절에서 `Residual → LayerNorm` 순서를 썼다. 이것이 원 논문 방식(**Post-LN**)이다.

최근 모델들은 **Pre-LN**을 많이 쓴다. LayerNorm을 먼저 적용하는 방식이다.

| 방식 | 구조 | 특징 |
|---|---|---|
| Post-LN | `x = Norm(x + Attn(x))` | 원 논문. 워밍업 없으면 학습이 불안정 |
| **Pre-LN** | `x = x + Attn(Norm(x))` | 학습이 안정적. 깊은 모델에 유리 |

**차이의 핵심**은 Residual 경로에 LayerNorm이 끼어 있느냐다.
Pre-LN에서는 `x`가 아무 변형 없이 끝까지 통과할 수 있다.

In [ ]:
import torch
import torch.nn as nn


class PreLNBlock(nn.Module):
    # Pre-LN 방식 인코더 블록

    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 정규화를 먼저 하고, Residual은 원본 x에 더한다
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


print("=" * 60)
print("Pre-LN vs Post-LN — 깊이에 따른 그래디언트")
print("=" * 60)


def grad_through_depth(block_cls, n_layers, d=64, seed=0):
    torch.manual_seed(seed)
    blocks = nn.ModuleList([block_cls(d, 4, dropout=0.0) for _ in range(n_layers)])
    x = torch.randn(1, 6, d, requires_grad=True)
    h = x
    for b in blocks:
        h = b(h)
    h.sum().backward()
    return x.grad.abs().mean().item()


print(f"{'층 수':<10}{'Post-LN':<20}{'Pre-LN':<20}{'비율'}")
print("-" * 60)
for n in [2, 4, 8, 16]:
    g_post = grad_through_depth(EncoderBlock, n)
    g_pre = grad_through_depth(PreLNBlock, n)
    print(f"{n:<10}{g_post:<20.4e}{g_pre:<20.4e}{g_pre/g_post:>8.2f}배")

print("-" * 60)
print()
print("Pre-LN 쪽이 깊어져도 그래디언트가 잘 유지된다.")
print("Residual 경로에 정규화가 끼어들지 않기 때문이다.")
print()
print("다만 Post-LN이 최종 성능은 더 좋다는 보고도 있어,")
print("워밍업(이론편 11.5절)을 충분히 주고 Post-LN을 쓰는 경우도 많다.")

---

## 9. 정리

### 만든 부품과 그 이유

| 부품 | 없으면 생기는 문제 | 확인한 방법 |
|---|---|---|
| Positional Encoding | 순서를 모른다 | 순서 뒤집어도 결과 동일 ✓ |
| Multi-Head | 한 종류 관계만 본다 | 헤드별 다른 패턴 ✓ |
| Feed-Forward | 토큰별 변환이 없다 | 토큰 독립성 확인 ✓ |
| Residual | 깊이 쌓으면 학습 불가 | 20층 그래디언트 비교 ✓ |
| LayerNorm | 학습 불안정 | BatchNorm과 대비 ✓ |

**부품마다 "왜 필요한가"를 먼저 확인**하고 붙인 것이 이 장의 방식이었다.

### 인코더 블록의 구조

```python
# Post-LN (원 논문)
x = norm1(x + attention(x))
x = norm2(x + ffn(x))

# Pre-LN (최근 방식)
x = x + attention(norm1(x))
x = x + ffn(norm2(x))
```

### 기억할 것

| 항목 | 요점 |
|---|---|
| PE | 차원마다 다른 주기의 파동 — 학습 불필요, 긴 문장에도 적용 |
| `contiguous()` | `transpose` 후 `view` 전에 필요 |
| FFN | 토큰끼리 섞지 않음 — 섞는 것은 Attention의 역할 |
| Residual | $\partial y/\partial x = 1 + \ldots$ 이라 그래디언트가 산다 |
| LayerNorm | 샘플 안에서 정규화 — 배치 크기와 무관 |
| 임베딩 × √d | 위치 인코딩과 크기를 맞추기 위해 |
| 파라미터 확인 | PyTorch 내장 구현과 개수가 일치 ✓ |

### 다음 장

**22. Transformer Decoder와 전체 모델** — 인코더에 두 가지를 더한다.

1. **Masked Self-Attention** — 미래를 못 보게 가린다
2. **Encoder-Decoder Attention** — 인코더의 정보를 참조한다

그리고 전체 모델을 만들어 실제 시퀀스 변환 과제를 학습시킨다.